# **Libra**
> Brady Spears, Los Alamos National Laboratory
## Serializing & Deserializing Schemas from a YAML File

---

### About this Notebook
This notebook provides insight into the simple instantiation of `Libra's` _Schema_ object, and gives examples of how to 'load' and 'dump' abstract SQLAlchemy _Table_ instances (referred to interchangably as _Models_) to and from specifically-formatted YAML files. Some `SQLAlchemy` boilerplate code is shown to exemplify how a user can interact with `SQLAlchemy` declartive objects, performing CRUD (create-remove-update-delete) operations in the OOP environment that are then reflected in the SQL environment. 

---

### Defining a Schema definition Dictionary
A "schema definition" is a serialized instance of a relational database schema, and in this case, is contained within a YAML file. Contained within the 
YAML file are 'columns' and 'models', which describe the structure of both `SQLAlchemy` _Column_ objects and abstract `SQLAlchemy` _Table_ objects. Key-value pairs contained in sub-dictionaries that describe each are ultimately passed to the '__init__()' methods of each object during their dynamic de-serialization. Using the _NNSA KB Core_, a "Knowledge Base" relational database schema developed by the National Nuclear Security Administration for purposes of seismic study, the serialized schema takes the form:

In [13]:
import yaml

yaml_filepath = 'data/kbcore.yaml'

with open(yaml_filepath, 'r') as file:
    schema_dict = yaml.safe_load(file)

print(schema_dict)

{'NNSA KB Core': {'description': 'National Nuclear Security Administration (NNSA) Knowledge Base (KB) Core relational database table schema', 'columns': {'algorithm': {'type': 'String(length = 15)', 'default': '-', 'info': {'format': '15.15s', 'width': 15, 'max_length': 15}}, 'amp': {'type': 'Float(precision = 24)', 'default': -1, 'info': {'format': '11.2f', 'width': 11, 'gt': 0}}, 'ampid': {'type': 'Numeric(9)', 'default': -1, 'info': {'format': '9d', 'width': 9, 'gt': 0}}, 'amptime': {'type': 'Float(precision = 53)', 'default': -9999999999.999, 'info': {'format': '17.5f', 'width': 17, 'ge': -9999999999.999}}, 'amptype': {'type': 'String(length = 8)', 'default': '-', 'info': {'format': '8.8s', 'width': 8, 'max_length': 8}}, 'arid': {'type': 'Numeric(9)', 'default': -1, 'info': {'format': '9d', 'width': 9, 'gt': 0}}, 'auth': {'type': 'String(length = 20)', 'default': '-', 'info': {'format': '20.20s', 'width': 20, 'max_length': 20}}, 'azdef': {'type': 'String(length = 1)', 'default': '-

Each 'column' entry must contain a 'type' key-value pair that maps to an SQLAlchemy _type_ object. Any extraneous information can be passed to a column's 'info' dictionary. In `Libra's` case, those values are exclusively used by various "mix-in" packages designed to augment model behavior in OOP space. 'Model' entries in the YAML file contain string references to each column that makes up that model as well as constraints. `Libra` supports Primary Key Constraints ('pk'), Unique Constraints ('uq'), Check Constraints ('ck'), and Indexes ('ix'). Foreign Key constraints are not currently supported due to restrictions on how `SQLAlchemy` internally resolves reference names when constructing _ForeignKeyConstraint_ objects. Future editions of `Libra` may implement support for Foreign keys, but do not in version 1.0.0.

We can use `Libra's` _Schema_ object to directly load in the table definitions from the YAML file:

In [14]:
from libra import Schema

kbcore = Schema('NNSA KB Core').load(yaml_filepath)

print(f'Schema \'{kbcore.name}\' contains :')
print(f'  Models : {list(kbcore._registry.models.keys())}')
print(f'  Columns : {list(kbcore._registry.columns.keys())}')

Schema 'NNSA KB Core' contains :
  Models : ['affiliation', 'amplitude', 'arrival', 'assoc', 'event', 'gregion', 'instrument', 'lastid', 'netmag', 'network', 'origerr', 'origin', 'remark', 'sensor', 'site', 'sitechan', 'sregion', 'stamag', 'wfdisc', 'wftag']
  Columns : ['algorithm', 'amp', 'ampid', 'amptime', 'amptype', 'arid', 'auth', 'azdef', 'azimuth', 'azres', 'band', 'belief', 'calib', 'calper', 'calratio', 'chan', 'chanid', 'clip', 'commid', 'conf', 'ctype', 'datatype', 'deast', 'delaz', 'delslo', 'delta', 'deltaf', 'deltim', 'depdp', 'depth', 'descrip', 'dfile', 'digital', 'dir', 'dnorth', 'dtype', 'duration', 'edepth', 'elev', 'ema', 'emares', 'endtime', 'esaz', 'etype', 'evid', 'evname', 'fm', 'foff', 'grn', 'grname', 'hang', 'inarrival', 'inid', 'insname', 'instant', 'instype', 'iphase', 'jdate', 'keyname', 'keyvalue', 'lat', 'lddate', 'lineno', 'logat', 'lon', 'magdef', 'magid', 'magnitude', 'magres', 'magtype', 'mb', 'mbid', 'ml', 'mlid', 'mmodel', 'ms', 'msid', 'nass', 'n

Through the _Schema_ object, concrete (not abstract) `SQLAlchemy` _Table_ objects can be created by simply creating a new class tha inherits from the Model and has a '__tablename__' attribute attached to it. The '__tablename__' attribute is the name of the table in the SQL environment and what `SQLAlchemy` uses to reference the table when rendering Data Definition Language (DDL). This DDL is emitted anytime a transaction with the database backend occurs, whether that is creating tables, dropping tables, inserting data, or updating data.

In [15]:
class Event(kbcore.event):
    __tablename__ = 'my_event_table'

# All methods and components of the Event object
print(Event.__table__.__dict__)

{'dispatch': <sqlalchemy.event.base.DDLEventsDispatch object at 0x7f340dfaf020>, 'name': 'my_event_table', '_columns': <sqlalchemy.sql.base.DedupeColumnCollection object at 0x7f340d1988c0>, 'primary_key': PrimaryKeyConstraint(Column('evid', Numeric(precision=9), table=<my_event_table>, primary_key=True, nullable=False, default=ScalarElementColumnDefault(-1))), 'foreign_keys': set(), 'fullname': 'my_event_table', 'metadata': MetaData(), 'schema': None, '_sentinel_column': None, 'indexes': set(), 'constraints': {UniqueConstraint(Column('prefor', Numeric(precision=9), table=<my_event_table>, default=ScalarElementColumnDefault(-1))), PrimaryKeyConstraint(Column('evid', Numeric(precision=9), table=<my_event_table>, primary_key=True, nullable=False, default=ScalarElementColumnDefault(-1)))}, 'c': <sqlalchemy.sql.base.ReadOnlyColumnCollection object at 0x7f340d3f5440>, '_extra_dependencies': set(), 'implicit_returning': True, 'comment': None, '_prefixes': [], 'description': 'my_event_table'}


Now that we have a concrete table, we can use standard `SQLAlchemy` declarative syntax to interact with it. Documentation on how to interact with declarative objects in `SQLAlchemy` is available at https://www.sqlalchemy.org/

In [16]:
import os
from datetime import datetime

import sqlalchemy
from sqlalchemy.orm import sessionmaker

local_db = 'data/examples.db' # Path to local sqlite3 database

# Remove local database for clean slate
if os.path.exists(local_db):
    os.remove(local_db)

engine = sqlalchemy.create_engine(f'sqlite:///{local_db}')
LocalSession = sessionmaker(bind = engine)

with LocalSession() as session:

    # Create all tables associated with kbcore_subset Schema object
    kbcore.base.metadata.create_all(engine)

    # Create a list of objects to add to the database
    entries = [
        Event(1, 'Earthquake in California', 1597, 'USGS', None, None),
        Event(2, None, 1599, 'UNR', 14, None),
        Event(evid = 3, evname = 'Eruption in Hawaii', prefor = 15, auth = 'USGS'),
        Event(evid = 4, prefor = 12, auth = 'ISC', lddate = datetime(2025, 1, 24))
    ]

    [session.add(entry) for entry in entries]
    session.commit() # Commit the additions to the database

    # Now, let's query all of that back
    query_results = session.query(Event).all()

    # Also, show an example of filtering
    filtered_query_results = session.query(Event).filter(Event.auth == 'USGS').all()

print('All query results:')
for row in query_results:
    print(f'  {row}')

print('Filtered query results:')
for row in filtered_query_results:
    print(f'  {row}')

All query results:
  Event(evid=1.0000000000, evname='Earthquake in California', prefor=1597.0000000000, auth='USGS', commid=-1.0000000000, lddate=2026-03-03 15:39:15.592273)
  Event(evid=2.0000000000, evname='-', prefor=1599.0000000000, auth='UNR', commid=14.0000000000, lddate=2026-03-03 15:39:15.592308)
  Event(evid=3.0000000000, evname='Eruption in Hawaii', prefor=15.0000000000, auth='USGS', commid=-1.0000000000, lddate=2026-03-03 15:39:15.592359)
  Event(evid=4.0000000000, evname='-', prefor=12.0000000000, auth='ISC', commid=-1.0000000000, lddate=2025-01-24 00:00:00)
Filtered query results:
  Event(evid=1.0000000000, evname='Earthquake in California', prefor=1597.0000000000, auth='USGS', commid=-1.0000000000, lddate=2026-03-03 15:39:15.592273)
  Event(evid=3.0000000000, evname='Eruption in Hawaii', prefor=15.0000000000, auth='USGS', commid=-1.0000000000, lddate=2026-03-03 15:39:15.592359)


Notice how above, in defining the _Event_ object, `Libra` supports both positional and keyword initialization patterns. For values that are _None_ or simply not included, `Libra's` custom metaclass defaults that column's value to the pre-defined 'default' parameter. 

`Libra` suppors the dumping of _Schema_ objects to YAML files, too. To export the loaded schema definition back to a YAML file:

In [17]:
import tempfile

temp_file = tempfile.NamedTemporaryFile(mode = 'w', suffix = '.yaml', delete = True)

kbcore.dump(temp_file.name)

# Reimport to ensure it was dumped properly:
kbcore = Schema('NNSA KB Core').load(temp_file.name)

print(f'Schema \'{kbcore.name}\' contains :')
print(f'  Models : {list(kbcore._registry.models.keys())}')
print(f'  Columns : {list(kbcore._registry.columns.keys())}')

Schema 'NNSA KB Core' contains :
  Models : ['affiliation', 'amplitude', 'arrival', 'assoc', 'event', 'gregion', 'instrument', 'lastid', 'netmag', 'network', 'origerr', 'origin', 'remark', 'sensor', 'site', 'sitechan', 'sregion', 'stamag', 'wfdisc', 'wftag']
  Columns : ['algorithm', 'amp', 'ampid', 'amptime', 'amptype', 'arid', 'auth', 'azdef', 'azimuth', 'azres', 'band', 'belief', 'calib', 'calper', 'calratio', 'chan', 'chanid', 'clip', 'commid', 'conf', 'ctype', 'datatype', 'deast', 'delaz', 'delslo', 'delta', 'deltaf', 'deltim', 'depdp', 'depth', 'descrip', 'dfile', 'digital', 'dir', 'dnorth', 'dtype', 'duration', 'edepth', 'elev', 'ema', 'emares', 'endtime', 'esaz', 'etype', 'evid', 'evname', 'fm', 'foff', 'grn', 'grname', 'hang', 'inarrival', 'inid', 'insname', 'instant', 'instype', 'iphase', 'jdate', 'keyname', 'keyvalue', 'lat', 'lddate', 'lineno', 'logat', 'lon', 'magdef', 'magid', 'magnitude', 'magres', 'magtype', 'mb', 'mbid', 'ml', 'mlid', 'mmodel', 'ms', 'msid', 'nass', 'n